In [1]:
# ----------------------------------------------------------
# STEP 1: TRAINING A REWARD MODEL (RM) FROM FEEDBACK LOGS
# ----------------------------------------------------------

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

# Load your feedback logs CSV (exported earlier from logging)
df = pd.read_csv(r"C:\Users\hp\Downloads\rlhf_feedback_log.csv")

# Drop rows without feedback
df = df.dropna(subset=["user_feedback"])

# Map feedback to binary score: Y = 1, N = 0
df["reward"] = df["user_feedback"].apply(lambda x: 1 if "Yes" in x else 0)
# Combine query + answer as the full input
df["text"] = df["query"] + " " + df["answer"]
# Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(), df["reward"].tolist(), test_size=0.2, random_state=42
)

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the text
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)


W0707 20:15:58.019000 2436 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
# Custom Dataset class
class RLDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        return {
            **{k: torch.tensor(v[idx]) for k, v in self.encodings.items()},
            "labels": torch.tensor(self.labels[idx])
        }
    def __len__(self):
        return len(self.labels)

train_dataset = RLDataset(train_encodings, train_labels)
val_dataset = RLDataset(val_encodings, val_labels)

In [3]:
# Load a pre-trained model for classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./test_trainer",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_dir="./logs",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,0.257202


Checkpoint destination directory ./test_trainer\checkpoint-1 already exists and is non-empty. Saving will proceed but saved results may be invalid.


TrainOutput(global_step=1, training_loss=0.419012188911438, metrics={'train_runtime': 13.8604, 'train_samples_per_second': 0.216, 'train_steps_per_second': 0.072, 'total_flos': 283666606560.0, 'train_loss': 0.419012188911438, 'epoch': 1.0})

In [4]:
# Save model for future use (scoring, reranking, fine-tuning)
model.save_pretrained("my_reward_model/")
tokenizer.save_pretrained("my_reward_model/")

('my_reward_model/tokenizer_config.json',
 'my_reward_model/special_tokens_map.json',
 'my_reward_model/vocab.txt',
 'my_reward_model/added_tokens.json')